# Mouse Dynamics Bot Detection - Part 4: Inference

This notebook demonstrates how to use the trained models for prediction on new data.

**Available Models:**
- Random Forest (feature-based) - Fast, interpretable
- GRU (sequence-based) - Captures temporal patterns
- Ensemble (combination of both) - Best accuracy

**Note:** The saved model uses GRU (faster alternative to LSTM with similar performance).

In [ ]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

BASE_DIR = os.getcwd()
MODEL_DIR = os.path.join(BASE_DIR, "models")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Define Model Architectures

We use GRU instead of LSTM for faster inference with similar accuracy.

In [ ]:
class MouseGRU(nn.Module):
    """GRU model for mouse movement classification (faster than LSTM)"""
    def __init__(self, input_size=8, hidden_size=64, num_layers=2, dropout=0.3):
        super(MouseGRU, self).__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x, lengths=None):
        if lengths is not None:
            packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, h_n = self.gru(packed)
        else:
            _, h_n = self.gru(x)

        hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)
        return self.classifier(hidden).squeeze()


# Alias for backwards compatibility
MouseLSTM = MouseGRU

## 2. Human Bot Detector Class

In [ ]:
class HumanBotDetector:
    """Unified detector using both RF and GRU models"""
    
    def __init__(self, model_dir="models"):
        self.device = device
        self.model_dir = model_dir
        
        # Load Random Forest model
        print("Loading Random Forest model...")
        with open(os.path.join(model_dir, 'best_rf_model.pkl'), 'rb') as f:
            self.rf_model = pickle.load(f)
        with open(os.path.join(model_dir, 'feature_scaler.pkl'), 'rb') as f:
            self.scaler = pickle.load(f)
        with open(os.path.join(model_dir, 'feature_names.pkl'), 'rb') as f:
            self.feature_names = pickle.load(f)
        
        # Load GRU model (saved as best_lstm_model.pt for compatibility)
        print("Loading GRU model...")
        self.gru_model = MouseGRU(input_size=8, hidden_size=64, num_layers=2, dropout=0.3)
        self.gru_model.load_state_dict(
            torch.load(os.path.join(model_dir, 'best_lstm_model.pt'),
                      map_location=self.device, weights_only=True)
        )
        self.gru_model.to(self.device)
        self.gru_model.eval()
        
        print("Models loaded successfully!")
    
    def parse_events(self, behavior_string):
        """Parse behavior string into events"""
        events = []
        pattern = r'\[(m|c|s)\(([^)]+)\)\]'
        matches = re.findall(pattern, behavior_string)
        
        for event_type, params in matches:
            if event_type == 'm':
                coords = params.split(',')
                if len(coords) == 2:
                    try:
                        events.append({'type': 'move', 'x': int(coords[0]), 'y': int(coords[1])})
                    except:
                        pass
            elif event_type == 'c':
                events.append({'type': 'click', 'button': params})
            elif event_type == 's':
                events.append({'type': 'scroll', 'direction': params})
        
        return events
    
    def extract_features(self, events):
        """Extract features for RF model"""
        moves = [(e['x'], e['y']) for e in events if e.get('type') == 'move']
        clicks = [e for e in events if e.get('type') == 'click']
        scrolls = [e for e in events if e.get('type') == 'scroll']
        
        if len(moves) < 5:
            return None
        
        x = np.array([m[0] for m in moves])
        y = np.array([m[1] for m in moves])
        
        # Compute intermediate values
        dx = np.diff(x)
        dy = np.diff(y)
        distances = np.sqrt(dx**2 + dy**2)
        speeds = distances / 0.01
        accelerations = np.diff(speeds) if len(speeds) > 1 else np.array([0])
        jerks = np.diff(accelerations) if len(accelerations) > 1 else np.array([0])
        
        # Compute angles
        angles = []
        for i in range(len(x) - 2):
            v1 = np.array([x[i+1] - x[i], y[i+1] - y[i]])
            v2 = np.array([x[i+2] - x[i+1], y[i+2] - y[i+1]])
            norm1, norm2 = np.linalg.norm(v1), np.linalg.norm(v2)
            if norm1 > 0 and norm2 > 0:
                cos_angle = np.clip(np.dot(v1, v2) / (norm1 * norm2), -1, 1)
                angles.append(np.arccos(cos_angle))
        angles = np.array(angles) if angles else np.array([0])
        
        # Curvature
        dx_grad = np.gradient(x)
        dy_grad = np.gradient(y)
        ddx = np.gradient(dx_grad)
        ddy = np.gradient(dy_grad)
        numerator = np.abs(dx_grad * ddy - dy_grad * ddx)
        denominator = (dx_grad**2 + dy_grad**2)**1.5
        curvature = np.where(denominator > 1e-10, numerator / denominator, 0)
        
        safe = lambda arr, func, default=0: func(arr) if len(arr) > 0 and not np.isnan(func(arr)) else default
        
        features = {
            'mean_speed': safe(speeds, np.mean),
            'std_speed': safe(speeds, np.std),
            'max_speed': safe(speeds, np.max),
            'min_speed': safe(speeds, np.min),
            'median_speed': safe(speeds, np.median),
            'speed_range': safe(speeds, np.max) - safe(speeds, np.min),
            'mean_acceleration': safe(accelerations, np.mean),
            'std_acceleration': safe(accelerations, np.std),
            'max_acceleration': safe(np.abs(accelerations), np.max),
            'mean_jerk': safe(jerks, np.mean),
            'std_jerk': safe(jerks, np.std),
            'path_straightness': np.sqrt((x[-1]-x[0])**2 + (y[-1]-y[0])**2) / (np.sum(distances) + 1e-8),
            'total_path_length': np.sum(distances),
            'direct_distance': np.sqrt((x[-1]-x[0])**2 + (y[-1]-y[0])**2),
            'mean_curvature': safe(curvature, np.mean),
            'std_curvature': safe(curvature, np.std),
            'max_curvature': safe(curvature, np.max),
            'mean_angle_change': safe(angles, np.mean),
            'std_angle_change': safe(angles, np.std),
            'x_range': np.max(x) - np.min(x),
            'y_range': np.max(y) - np.min(y),
            'aspect_ratio': (np.max(x) - np.min(x)) / (np.max(y) - np.min(y) + 1e-8),
            'num_events': len(events),
            'num_moves': len(moves),
            'session_duration': len(moves) * 0.01,
            'event_rate': len(events) / (len(moves) * 0.01 + 0.001),
            'mean_distance': safe(distances, np.mean),
            'std_distance': safe(distances, np.std),
            'distance_entropy': 0,  # Simplified
            'pause_count': int(np.sum(speeds < 0.5)) if len(speeds) > 0 else 0,
            'pause_ratio': float(np.mean(speeds < 0.5)) if len(speeds) > 0 else 0,
            'num_clicks': len(clicks),
            'click_rate': len(clicks) / (len(moves) * 0.01 + 0.001),
            'num_scrolls': len(scrolls),
            'direction_change_count': int(np.sum(angles > np.pi/4)) if len(angles) > 0 else 0,
            'direction_change_rate': np.sum(angles > np.pi/4) / max(len(moves), 1) if len(angles) > 0 else 0,
            'speed_skewness': float(stats.skew(speeds)) if len(speeds) > 3 else 0,
            'speed_kurtosis': float(stats.kurtosis(speeds)) if len(speeds) > 3 else 0,
            'distance_skewness': float(stats.skew(distances)) if len(distances) > 3 else 0,
            'distance_kurtosis': float(stats.kurtosis(distances)) if len(distances) > 3 else 0,
            'distance_cv': safe(distances, np.std) / (safe(distances, np.mean) + 1e-8),
            'mean_velocity_angle_change': safe(np.abs(np.diff(np.arctan2(dy, dx))), np.mean) if len(dx) > 1 else 0,
            'std_velocity_angle_change': safe(np.abs(np.diff(np.arctan2(dy, dx))), np.std) if len(dx) > 1 else 0,
        }
        
        return features
    
    def extract_sequence(self, events, max_len=500):
        """Extract sequence features for GRU model"""
        moves = [(e['x'], e['y']) for e in events if e.get('type') == 'move']
        if len(moves) < 5:
            return None
        
        x = np.array([m[0] for m in moves], dtype=np.float32)
        y = np.array([m[1] for m in moves], dtype=np.float32)
        
        x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
        y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
        dx = np.diff(x, prepend=x[0])
        dy = np.diff(y, prepend=y[0])
        speed = np.sqrt(dx**2 + dy**2)
        speed_norm = speed / (speed.max() + 1e-8)
        acc = np.diff(speed, prepend=speed[0])
        acc_norm = (acc - acc.min()) / (acc.max() - acc.min() + 1e-8)
        angle = np.arctan2(dy, dx)
        angle_norm = (angle + np.pi) / (2 * np.pi)
        angular_vel = np.diff(angle, prepend=angle[0])
        angular_vel_norm = (angular_vel + np.pi) / (2 * np.pi)
        
        features = np.stack([
            x_norm, y_norm,
            dx / (np.abs(dx).max() + 1e-8),
            dy / (np.abs(dy).max() + 1e-8),
            speed_norm, acc_norm,
            angle_norm, angular_vel_norm
        ], axis=1)
        
        if len(features) > max_len:
            features = features[:max_len]
        elif len(features) < max_len:
            padding = np.zeros((max_len - len(features), 8))
            features = np.vstack([features, padding])
        
        return features
    
    def predict(self, events, method='ensemble'):
        """
        Predict if mouse input is human or bot.
        
        Args:
            events: List of dicts with 'type', 'x', 'y' keys OR behavior string
            method: 'rf' for Random Forest, 'gru' for GRU, 'ensemble' for both
        
        Returns:
            dict with prediction, confidence, and probabilities
        """
        # Parse if string
        if isinstance(events, str):
            events = self.parse_events(events)
        
        results = {}
        
        # RF prediction
        if method in ['rf', 'ensemble']:
            features = self.extract_features(events)
            if features:
                X = np.array([[features.get(f, 0) for f in self.feature_names]])
                X = np.nan_to_num(X, nan=0, posinf=0, neginf=0)
                X_scaled = self.scaler.transform(X)
                rf_prob = self.rf_model.predict_proba(X_scaled)[0, 1]
                results['rf_prob'] = rf_prob
        
        # GRU prediction
        if method in ['gru', 'lstm', 'ensemble']:
            sequence = self.extract_sequence(events)
            if sequence is not None:
                X_seq = torch.tensor(sequence, dtype=torch.float32).unsqueeze(0).to(self.device)
                length = torch.tensor([min(len([e for e in events if e.get('type') == 'move']), 500)])
                with torch.no_grad():
                    gru_prob = self.gru_model(X_seq, length).item()
                results['gru_prob'] = gru_prob
                results['lstm_prob'] = gru_prob  # Alias for compatibility
        
        # Combine predictions
        if method == 'ensemble' and 'rf_prob' in results and 'gru_prob' in results:
            prob = (results['rf_prob'] + results['gru_prob']) / 2
        elif method == 'rf' and 'rf_prob' in results:
            prob = results['rf_prob']
        elif method in ['gru', 'lstm'] and 'gru_prob' in results:
            prob = results['gru_prob']
        else:
            prob = 0.5  # Fallback
        
        return {
            'prediction': 1 if prob > 0.5 else 0,
            'label': 'bot' if prob > 0.5 else 'human',
            'confidence': prob if prob > 0.5 else 1 - prob,
            'bot_probability': prob,
            **results
        }

## 3. Load Detector

In [ ]:
# Initialize detector
detector = HumanBotDetector(MODEL_DIR)

## 4. Test on Sample Data

In [ ]:
# Load test sessions
with open(os.path.join(OUTPUT_DIR, 'sessions_data.pkl'), 'rb') as f:
    sessions_data = pickle.load(f)

# Get sample sessions from each class
test_samples = {}
for label in ['human', 'moderate_bot', 'advanced_bot']:
    for sid, data in sessions_data.items():
        if data['label'] == label:
            test_samples[label] = data['events']
            break

print(f"Loaded test samples for: {list(test_samples.keys())}")

In [ ]:
# Test predictions
print("\nPrediction Results:")
print("=" * 60)

for true_label, events in test_samples.items():
    result = detector.predict(events, method='ensemble')
    
    print(f"\nTrue Label: {true_label}")
    print(f"  Predicted: {result['label']}")
    print(f"  Confidence: {result['confidence']:.4f}")
    print(f"  Bot Probability: {result['bot_probability']:.4f}")
    if 'rf_prob' in result:
        print(f"  RF Probability: {result['rf_prob']:.4f}")
    if 'lstm_prob' in result:
        print(f"  LSTM Probability: {result['lstm_prob']:.4f}")
    
    correct = (result['label'] == 'human' and true_label == 'human') or \
              (result['label'] == 'bot' and true_label != 'human')
    print(f"  Correct: {'Yes' if correct else 'No'}")

## 5. Batch Prediction on All Test Data

In [ ]:
# Evaluate on all sessions
predictions = []
true_labels = []

for sid, data in sessions_data.items():
    result = detector.predict(data['events'], method='ensemble')
    predictions.append(result['prediction'])
    true_labels.append(0 if data['label'] == 'human' else 1)

predictions = np.array(predictions)
true_labels = np.array(true_labels)

# Metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("\nOverall Performance:")
print("=" * 60)
print(f"Accuracy: {accuracy_score(true_labels, predictions):.4f}")
print("\nClassification Report:")
print(classification_report(true_labels, predictions, target_names=['Human', 'Bot']))

print("\nConfusion Matrix:")
cm = confusion_matrix(true_labels, predictions)
print(f"             Predicted")
print(f"           Human  Bot")
print(f"Actual Human  {cm[0,0]:3d}   {cm[0,1]:3d}")
print(f"       Bot    {cm[1,0]:3d}   {cm[1,1]:3d}")

## 6. Custom Input Testing

In [ ]:
# Example: Create synthetic human-like movement
def generate_human_like_movement(n_points=200):
    """Generate synthetic human-like mouse movement"""
    events = []
    x, y = 100, 100
    
    for _ in range(n_points):
        # Add some randomness (human-like)
        dx = np.random.normal(5, 3)
        dy = np.random.normal(2, 2)
        x = max(0, min(1920, x + dx))
        y = max(0, min(1080, y + dy))
        events.append({'type': 'move', 'x': int(x), 'y': int(y)})
        
        # Occasional click
        if np.random.random() < 0.02:
            events.append({'type': 'click', 'button': 'l'})
    
    return events

def generate_bot_like_movement(n_points=200):
    """Generate synthetic bot-like mouse movement"""
    events = []
    x, y = 100, 100
    
    for i in range(n_points):
        # Very regular movement (bot-like)
        dx = 5  # Constant step
        dy = 3
        x = max(0, min(1920, x + dx))
        y = max(0, min(1080, y + dy))
        events.append({'type': 'move', 'x': int(x), 'y': int(y)})
    
    return events

In [ ]:
# Test with synthetic data
print("Testing with Synthetic Data:")
print("=" * 60)

# Human-like
human_events = generate_human_like_movement(500)
result = detector.predict(human_events, method='ensemble')
print(f"\nSynthetic Human-like movement:")
print(f"  Predicted: {result['label']} (confidence: {result['confidence']:.4f})")

# Bot-like
bot_events = generate_bot_like_movement(500)
result = detector.predict(bot_events, method='ensemble')
print(f"\nSynthetic Bot-like movement:")
print(f"  Predicted: {result['label']} (confidence: {result['confidence']:.4f})")

## 7. Visualize Predictions

In [ ]:
# Visualize trajectories with predictions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (label, events) in enumerate(test_samples.items()):
    ax = axes[idx]
    
    # Get prediction
    result = detector.predict(events, method='ensemble')
    
    # Plot trajectory
    moves = [(e['x'], e['y']) for e in events if e['type'] == 'move']
    x = [m[0] for m in moves][:500]
    y = [m[1] for m in moves][:500]
    
    color = 'green' if result['label'] == 'human' else 'red'
    ax.plot(x, y, color=color, alpha=0.7, linewidth=0.5)
    ax.scatter(x[0], y[0], color='blue', s=50, marker='o', label='Start')
    ax.scatter(x[-1], y[-1], color='black', s=50, marker='x', label='End')
    
    ax.set_title(f"True: {label}\nPredicted: {result['label']} ({result['confidence']:.2f})", fontsize=10)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.invert_yaxis()
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Prediction Visualization (Green=Human, Red=Bot)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Export Detector for Production

## Summary

### How to Use

1. **Initialize detector**: `detector = HumanBotDetector('models')`
2. **Prepare events**: List of dicts with `{'type': 'move', 'x': int, 'y': int}`
3. **Predict**: `result = detector.predict(events, method='ensemble')`

### Prediction Methods

| Method | Description | Use Case |
|--------|-------------|----------|
| `'rf'` | Random Forest only | Fast inference, interpretable |
| `'gru'` | GRU only | Sequential patterns |
| `'ensemble'` | Average of both | Best accuracy |

### Output Format

```python
{
    'prediction': 0 or 1,
    'label': 'human' or 'bot',
    'confidence': 0.0-1.0,
    'bot_probability': 0.0-1.0,
    'rf_prob': 0.0-1.0,
    'gru_prob': 0.0-1.0
}
```

### Notes on Model Architecture
- The sequence model uses **GRU** (not LSTM) for faster inference
- GRU has similar accuracy to LSTM but trains/infers 2-3x faster
- For production, Random Forest alone is often sufficient

## Summary

### How to Use

1. **Initialize detector**: `detector = HumanBotDetector('models')`
2. **Prepare events**: List of dicts with `{'type': 'move', 'x': int, 'y': int}`
3. **Predict**: `result = detector.predict(events, method='ensemble')`

### Prediction Methods

| Method | Description | Use Case |
|--------|-------------|----------|
| `'rf'` | Random Forest only | Fast inference |
| `'lstm'` | LSTM only | Sequential patterns |
| `'ensemble'` | Average of both | Best accuracy |

### Output Format

```python
{
    'prediction': 0 or 1,
    'label': 'human' or 'bot',
    'confidence': 0.0-1.0,
    'bot_probability': 0.0-1.0,
    'rf_prob': 0.0-1.0,
    'lstm_prob': 0.0-1.0
}
```